# Automatic segmentation từ ảnh (COLMAP) → Detection3D

Pipeline thay thế cho việc segment tay bằng CloudCompare (`manual_segmentation.py`), dùng khi đầu vào là **ảnh chụp đa góc** đã dựng model 3D bằng COLMAP, thay vì máy scan.

Các bước:
1. Đọc pose camera + intrinsics từ COLMAP (`sparse` model) - code đọc COLMAP viết ngay trong notebook, không phụ thuộc `pycolmap`.
2. Chạy segment 2D (Grounding DINO + SAM2, y hệt `door_window_segmentation_in_2D.ipynb`) trên **các ảnh key-frame đã chọn tay** (`SELECTED_IMAGES` ở cell config bên dưới).
3. Với mỗi mask 2D, chiếu ngược từng pixel lên 3D bằng depth map dense của COLMAP + pose camera đó (unproject viết ngay trong notebook).
4. Gộp (merge) các instance của cùng 1 vật thể vật lý được nhìn thấy từ nhiều ảnh khác nhau.
5. Với mỗi cụm đã gộp, tìm mặt tường gần nhất trên toàn bộ point cloud của scene (dùng lại `estimate_up_vector_manhattan`/`extract_wall_planes` có sẵn trong `plane_fitting.py`; phần orient-tường-ra-ngoài + đo kích thước viết lại ngay trong notebook, không đụng vào `manual_segmentation.py`/`matching/` đang hoạt động đúng) để dựng `Detection3D` (center, u_axis, v_axis, normal, width, height).
6. Xuất ra: (a) file `.pkl` chứa `list[Detection3D]` để đưa thẳng vào bước matching (`sgd_alignment.matching`, không sửa gì phần này), và (b) 1 file `.ply` tô màu theo category để kiểm tra bằng mắt.

Chạy notebook này **riêng cho từng phía** (indoor và outdoor là 2 lần COLMAP/2 bộ ảnh khác nhau) bằng cách đổi `IS_OUTDOOR` + các đường dẫn ở cell config, giống hệt cách `manual_segmentation.py` được gọi 2 lần (`is_outdoor=False` rồi `is_outdoor=True`).

> Toàn bộ logic mới (đọc COLMAP, chiếu 2D→3D, gộp instance, đo opening) nằm gọn trong notebook này; không file `.py` nào trong `src/` bị thêm/sửa.

## Các file bạn cần chuẩn bị (input)

| Biến | Nội dung | Ghi chú |
|---|---|---|
| `SPARSE_DIR` | thư mục COLMAP sparse model: `cameras.bin`+`images.bin` hoặc `cameras.txt`+`images.txt` | **Phải là model đã undistort** (`colmap image_undistorter`), vì `colmap_io.Camera.intrinsics_matrix()` chỉ hỗ trợ camera không còn distortion (PINHOLE/SIMPLE_PINHOLE) |
| `IMAGES_DIR` | thư mục ảnh khớp với `SPARSE_DIR` ở trên | tức là `<dense>/images`, ảnh đã undistort - **không phải** ảnh gốc |
| `DEPTH_MAPS_DIR` | thư mục depth map dense: `<dense>/stereo/depth_maps/` | chứa file `<tên_ảnh>.geometric.bin` (hoặc `.photometric.bin`) sinh ra bởi `colmap patch_match_stereo` |
| `SCENE_PLY` | 1 file `.ply` (x, y, z) của toàn bộ point cloud dense đã fuse (`colmap stereo_fusion`, hoặc dense point cloud bạn export) | dùng để ước lượng trục "lên" + tìm mặt tường, giống input của `manual_segmentation.py` |
| `SELECTED_IMAGES` | danh sách tên file ảnh (key-frame) bạn chọn tay để segment | phải khớp tên trong `images.bin/.txt`, ví dụ `["frame_0012.jpg", "frame_0045.jpg"]` |


In [ ]:
%pip install -q opencv-python matplotlib pillow transformers timm accelerate torch torchvision ultralytics


In [ ]:
import os
import sys
import pickle

import numpy as np

sys.path.insert(0, os.path.abspath("src"))


## Cấu hình (điền đường dẫn thật của bạn vào đây)

In [ ]:
# --- COLMAP dense workspace (undistorted) ---
SPARSE_DIR = "data/colmap/dense/sparse"          # .bin/.txcamerast + images.bin/.txt
IMAGES_DIR = "data/colmap/dense/images"           # ảnh undistorted khớp với SPARSE_DIR
DEPTH_MAPS_DIR = "data/colmap/dense/stereo/depth_maps"  # <ten_anh>.geometric.bin

# point cloud toàn scene (đã fuse), dùng để ước lượng trục lên + mặt tường
SCENE_PLY = "data/colmap/dense/fused.ply"

# ảnh key-frame chọn tay để segment (tên phải khớp images.bin/.txt)
SELECTED_IMAGES = [
    # "frame_0012.jpg",
    # "frame_0045.jpg",
]

IS_OUTDOOR = False  # đổi thành True khi chạy cho phía outdoor

OUTPUT_DIR = "outputs"
OUTPUT_NAME = "outdoor" if IS_OUTDOOR else "indoor"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ngưỡng gộp instance của cùng 1 vật thể nhìn thấy từ nhiều ảnh khác nhau
# (centroid 2 instance cùng category cách nhau dưới ngưỡng này -> coi là 1 vật thể)
MERGE_DISTANCE = 0.6  # mét

# lọc điểm depth không hợp lệ khi chiếu ngược 2D->3D
DEPTH_RANGE = (0.05, 30.0)  # mét


## Bước 1: đọc pose camera + intrinsics từ COLMAP

Đọc trực tiếp `cameras.bin/.txt` + `images.bin/.txt` của COLMAP (tự nhận diện text/binary), không phụ thuộc `pycolmap`.

Quy ước COLMAP: 1 điểm world `X_world` chiếu vào không gian camera bằng `X_cam = R @ X_world + t`, với `R = qvec2rotmat(qvec)`. Depth map dense của `patch_match_stereo` lưu **Z-depth trong hệ camera** (không phải khoảng cách theo tia), nên phép chiếu ngược ở Bước 3 chỉ là pinhole nghịch đảo thông thường.

**Lưu ý quan trọng**: `SPARSE_DIR`/`IMAGES_DIR`/`DEPTH_MAPS_DIR` phải trỏ vào dense workspace đã **undistort** (`colmap image_undistorter`), vì `intrinsics_matrix()` bên dưới chỉ hỗ trợ camera không còn distortion (PINHOLE/SIMPLE_PINHOLE) - đây chính là output chuẩn của bước undistort.


In [ ]:
import struct
from dataclasses import dataclass
from pathlib import Path

# model_id -> (model_name, num_params), theo COLMAP src/colmap/scene/camera_model.h
_CAMERA_MODELS = {
    0: ("SIMPLE_PINHOLE", 3), 1: ("PINHOLE", 4), 2: ("SIMPLE_RADIAL", 4),
    3: ("RADIAL", 5), 4: ("OPENCV", 8), 5: ("OPENCV_FISHEYE", 8),
    6: ("FULL_OPENCV", 12), 7: ("FOV", 5), 8: ("SIMPLE_RADIAL_FISHEYE", 4),
    9: ("RADIAL_FISHEYE", 5), 10: ("THIN_PRISM_FISHEYE", 12),
}


@dataclass
class Camera:
    id: int
    model: str
    width: int
    height: int
    params: np.ndarray

    def intrinsics_matrix(self) -> np.ndarray:
        if self.model == "PINHOLE":
            fx, fy, cx, cy = self.params
        elif self.model == "SIMPLE_PINHOLE":
            f, cx, cy = self.params
            fx = fy = f
        else:
            raise ValueError(
                f"camera {self.id} có model {self.model} (còn distortion). "
                "Hãy chạy `colmap image_undistorter` trước và trỏ SPARSE_DIR/IMAGES_DIR/"
                "DEPTH_MAPS_DIR vào output của bước đó."
            )
        return np.array([[fx, 0.0, cx], [0.0, fy, cy], [0.0, 0.0, 1.0]])


@dataclass
class ColmapImage:
    id: int
    qvec: np.ndarray  # (4,) qw, qx, qy, qz
    tvec: np.ndarray  # (3,)
    camera_id: int
    name: str

    def rotation_matrix(self) -> np.ndarray:
        return qvec2rotmat(self.qvec)


def qvec2rotmat(qvec: np.ndarray) -> np.ndarray:
    qw, qx, qy, qz = qvec
    return np.array([
        [1 - 2 * qy**2 - 2 * qz**2, 2 * qx * qy - 2 * qz * qw, 2 * qx * qz + 2 * qy * qw],
        [2 * qx * qy + 2 * qz * qw, 1 - 2 * qx**2 - 2 * qz**2, 2 * qy * qz - 2 * qx * qw],
        [2 * qx * qz - 2 * qy * qw, 2 * qy * qz + 2 * qx * qw, 1 - 2 * qx**2 - 2 * qy**2],
    ])


def _read_cameras_text(path):
    cameras = {}
    for line in Path(path).read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        parts = line.split()
        cam_id, model, width, height = int(parts[0]), parts[1], int(parts[2]), int(parts[3])
        cameras[cam_id] = Camera(id=cam_id, model=model, width=width, height=height,
                                  params=np.array([float(p) for p in parts[4:]]))
    return cameras


def _read_cameras_binary(path):
    cameras = {}
    with open(path, "rb") as f:
        num_cameras = struct.unpack("<Q", f.read(8))[0]
        for _ in range(num_cameras):
            cam_id, model_id, width, height = struct.unpack("<iiQQ", f.read(24))
            model_name, num_params = _CAMERA_MODELS[model_id]
            params = np.array(struct.unpack("<" + "d" * num_params, f.read(8 * num_params)))
            cameras[cam_id] = Camera(id=cam_id, model=model_name, width=width, height=height, params=params)
    return cameras


def _read_images_text(path):
    images = {}
    lines = [l.strip() for l in Path(path).read_text().splitlines() if l.strip() and not l.startswith("#")]
    for i in range(0, len(lines), 2):  # mỗi ảnh chiếm 2 dòng (pose, rồi danh sách điểm 2D)
        parts = lines[i].split()
        img_id = int(parts[0])
        qvec = np.array([float(p) for p in parts[1:5]])
        tvec = np.array([float(p) for p in parts[5:8]])
        images[img_id] = ColmapImage(id=img_id, qvec=qvec, tvec=tvec, camera_id=int(parts[8]), name=parts[9])
    return images


def _read_images_binary(path):
    images = {}
    with open(path, "rb") as f:
        num_images = struct.unpack("<Q", f.read(8))[0]
        for _ in range(num_images):
            img_id = struct.unpack("<I", f.read(4))[0]
            qvec = np.array(struct.unpack("<dddd", f.read(32)))
            tvec = np.array(struct.unpack("<ddd", f.read(24)))
            cam_id = struct.unpack("<I", f.read(4))[0]
            name = b""
            while True:
                c = f.read(1)
                if c == b"\x00":
                    break
                name += c
            num_points2d = struct.unpack("<Q", f.read(8))[0]
            f.read(24 * num_points2d)  # bỏ qua các cặp (x, y, point3D_id)
            images[img_id] = ColmapImage(id=img_id, qvec=qvec, tvec=tvec, camera_id=cam_id, name=name.decode("utf-8"))
    return images


def read_colmap_model(sparse_dir):
    sparse_dir = Path(sparse_dir)
    if (sparse_dir / "cameras.bin").exists():
        return _read_cameras_binary(sparse_dir / "cameras.bin"), _read_images_binary(sparse_dir / "images.bin")
    if (sparse_dir / "cameras.txt").exists():
        return _read_cameras_text(sparse_dir / "cameras.txt"), _read_images_text(sparse_dir / "images.txt")
    raise FileNotFoundError(f"không tìm thấy cameras.bin/.txt trong {sparse_dir}")


def image_by_name(images, name):
    for img in images.values():
        if img.name == name or Path(img.name).name == Path(name).name:
            return img
    raise KeyError(f"không tìm thấy ảnh {name!r} trong COLMAP model")


cameras, images = read_colmap_model(SPARSE_DIR)
print(f"{len(cameras)} camera(s), {len(images)} ảnh trong model")

selected_images_info = []
for name in SELECTED_IMAGES:
    img = image_by_name(images, name)
    cam = cameras[img.camera_id]
    print(f"  {name}: camera model={cam.model}, {cam.width}x{cam.height}")
    selected_images_info.append((name, img, cam))


## Bước 2: segment 2D trên các ảnh đã chọn

Y hệt `door_window_segmentation_in_2D.ipynb` (Grounding DINO detect box theo text prompt "door. window." + SAM2 segment theo box), chỉ khác là giữ lại **mask nhị phân** của từng instance (cần cho bước chiếu 3D) thay vì chỉ vẽ overlay.


In [ ]:
import cv2
import torch
from PIL import Image as PILImage
from ultralytics import SAM
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

TEXT_PROMPT = "door. window."
BOX_THRESHOLD = 0.5
TEXT_THRESHOLD = 0.25
NMS_IOU_THRES = 0.7
CROSS_LABEL_IOU_THRES = 0.7

DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

GDINO_MODEL_ID = "IDEA-Research/grounding-dino-base"
gdino_processor = AutoProcessor.from_pretrained(GDINO_MODEL_ID)
gdino_model = AutoModelForZeroShotObjectDetection.from_pretrained(GDINO_MODEL_ID).to(DEVICE)
segmenter = SAM("sam2.1_b.pt")


In [ ]:
CLASS_COLORS = {"door": (60, 180, 75), "window": (255, 130, 0)}


def canonical_label(text_label: str):
    l = text_label.lower()
    if "door" in l:
        return "door"
    if "window" in l:
        return "window"
    return None


def _iou(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])
    return inter / (area_a + area_b - inter + 1e-9)


def detect_boxes(image_rgb: np.ndarray):
    pil_image = PILImage.fromarray(image_rgb)
    inputs = gdino_processor(images=pil_image, text=TEXT_PROMPT, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = gdino_model(**inputs)
    result = gdino_processor.post_process_grounded_object_detection(
        outputs, inputs.input_ids,
        threshold=BOX_THRESHOLD, text_threshold=TEXT_THRESHOLD,
        target_sizes=[pil_image.size[::-1]],
    )[0]

    boxes, scores, labels = [], [], []
    for box, score, text_label in zip(result["boxes"], result["scores"], result["labels"]):
        label = canonical_label(text_label)
        if label is not None:
            boxes.append(box.tolist())
            scores.append(float(score))
            labels.append(label)

    if not boxes:
        return np.zeros((0, 4)), np.array([]), np.array([])

    boxes, scores, labels = np.array(boxes), np.array(scores), np.array(labels)

    keep = []
    for label in np.unique(labels):
        idxs = np.where(labels == label)[0]
        xywh = [[x1, y1, x2 - x1, y2 - y1] for x1, y1, x2, y2 in boxes[idxs].tolist()]
        nms_idx = cv2.dnn.NMSBoxes(xywh, scores[idxs].tolist(), score_threshold=0.0, nms_threshold=NMS_IOU_THRES)
        keep.extend(idxs[np.array(nms_idx).flatten()])
    keep = np.array(keep)
    boxes, scores, labels = boxes[keep], scores[keep], labels[keep]

    order = np.argsort(scores)[::-1]
    final = []
    for i in order:
        if all(_iou(boxes[i], boxes[j]) < CROSS_LABEL_IOU_THRES for j in final):
            final.append(i)
    final = np.array(final)
    return boxes[final], scores[final], labels[final]


def segment_image(image_path: str, out_dir: str):
    """Same as `run_segmentation` in door_window_segmentation_in_2D.ipynb,
    but also returns each instance's binary mask (needed for 3D backprojection)."""
    image_bgr = cv2.imread(image_path)
    if image_bgr is None:
        raise FileNotFoundError(image_path)
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    overlay = image_bgr.copy()
    h, w = image_bgr.shape[:2]

    boxes, scores, labels = detect_boxes(image_rgb)
    instances = []

    if len(boxes) > 0:
        sam_result = segmenter.predict(image_bgr, bboxes=boxes, verbose=False)[0]
        masks = sam_result.masks.data.cpu().numpy() if sam_result.masks is not None else []

        for box, label, conf, mask in zip(boxes, labels, scores, masks):
            color = CLASS_COLORS.get(label, (0, 0, 255))
            mask_bool = mask.astype(bool)
            if mask_bool.shape[:2] != (h, w):
                mask_bool = cv2.resize(
                    mask_bool.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST,
                ).astype(bool)

            overlay[mask_bool] = (0.5 * np.array(color) + 0.5 * overlay[mask_bool]).astype(np.uint8)
            x1, y1, x2, y2 = box.astype(int)
            cv2.rectangle(overlay, (x1, y1), (x2, y2), color, 2)
            cv2.putText(overlay, f"{label} {conf:.2f}", (x1, max(y1 - 8, 15)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2, cv2.LINE_AA)

            instances.append({"label": label, "conf": float(conf), "box": box.tolist(), "mask": mask_bool})

    out_path = os.path.join(out_dir, os.path.splitext(os.path.basename(image_path))[0] + "_segmented.jpg")
    cv2.imwrite(out_path, overlay)
    return overlay, instances, out_path


In [ ]:
import matplotlib.pyplot as plt

per_image_instances = {}  # image_name -> list of {label, conf, box, mask}
overlays = []
for name in SELECTED_IMAGES:
    image_path = os.path.join(IMAGES_DIR, name)
    overlay, instances, out_path = segment_image(image_path, OUTPUT_DIR)
    per_image_instances[name] = instances
    overlays.append((name, overlay))
    print(f"{name} -> {len(instances)} object(s), saved overlay to {out_path}")
    for inst in instances:
        print(f"    {inst['label']}: conf={inst['conf']:.2f}, box={inst['box']}")

fig, axes = plt.subplots(1, len(overlays), figsize=(6 * max(len(overlays), 1), 6))
if len(overlays) == 1:
    axes = [axes]
for ax, (name, overlay) in zip(axes, overlays):
    ax.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()
plt.show()


## Bước 3: chiếu từng mask 2D lên 3D (theo depth map dense của đúng ảnh đó)

`read_colmap_array` đọc depth/normal map dạng `.bin` của COLMAP (header ascii `width&height&channels&` rồi dữ liệu float32 column-major). Với mỗi instance: resize mask về đúng độ phân giải của depth map (thường bằng độ phân giải ảnh dense, có thể khác ảnh gốc), rồi unproject từng pixel có depth hợp lệ thành 1 điểm world bằng pinhole nghịch đảo + pose camera của đúng ảnh đó.


In [ ]:
def read_colmap_array(path):
    with open(path, "rb") as f:
        header = b""
        while True:
            c = f.read(1)
            if c == b"&" and header.count(b"&") == 2:
                break
            header += c
        width, height, channels = map(int, header.split(b"&"))
        data = np.fromfile(f, dtype=np.float32)
    data = data.reshape(channels, width, height).transpose(2, 1, 0)  # -> (H, W, C)
    return data[:, :, 0] if channels == 1 else data


def backproject_mask_to_world(mask, depth, camera, image, depth_range=DEPTH_RANGE):
    """Unproject mọi pixel trong mask có depth hợp lệ thành 1 điểm world.
    Quy ước COLMAP: X_cam = R @ X_world + t => X_world = R.T @ (X_cam - t).
    """
    if mask.shape != depth.shape:
        raise ValueError(f"mask shape {mask.shape} != depth shape {depth.shape}; resize mask trước")

    valid = mask.astype(bool) & (depth > depth_range[0]) & (depth < depth_range[1])
    ys, xs = np.nonzero(valid)
    if len(xs) == 0:
        return np.zeros((0, 3))

    K = camera.intrinsics_matrix()
    fx, fy, cx, cy = K[0, 0], K[1, 1], K[0, 2], K[1, 2]
    d = depth[ys, xs]
    x_cam = (xs - cx) / fx * d
    y_cam = (ys - cy) / fy * d
    points_cam = np.stack([x_cam, y_cam, d], axis=1)

    R = image.rotation_matrix()
    t = image.tvec
    return (points_cam - t) @ R  # = R.T @ (points_cam - t), theo hàng


raw_instances = []  # list of {label, conf, source_image, points (N,3)}
for name, img, cam in selected_images_info:
    depth_path = os.path.join(DEPTH_MAPS_DIR, name + ".geometric.bin")
    if not os.path.exists(depth_path):
        depth_path = os.path.join(DEPTH_MAPS_DIR, name + ".photometric.bin")
    depth = read_colmap_array(depth_path)

    for inst in per_image_instances[name]:
        mask = inst["mask"]
        if mask.shape != depth.shape:
            mask = cv2.resize(mask.astype(np.uint8), (depth.shape[1], depth.shape[0]),
                               interpolation=cv2.INTER_NEAREST).astype(bool)
        points = backproject_mask_to_world(mask, depth, cam, img, depth_range=DEPTH_RANGE)
        print(f"{name}: {inst['label']} (conf {inst['conf']:.2f}) -> {len(points)} điểm 3D hợp lệ")
        if len(points) == 0:
            continue
        raw_instances.append({
            "label": inst["label"], "conf": inst["conf"], "source_image": name, "points": points,
        })


## Bước 4: gộp các instance của cùng 1 vật thể thấy từ nhiều ảnh

Gộp tham lam (greedy): cùng category + centroid cách nhau dưới `MERGE_DISTANCE` -> coi là cùng 1 vật thể, cộng dồn điểm. Đây là bước "instance merging" đã bàn khi thiết kế pipeline - cần thiết vì cùng 1 cửa/sổ thường xuất hiện trong nhiều ảnh key-frame khác nhau.


In [ ]:
def merge_instances(instances, merge_distance=MERGE_DISTANCE):
    clusters = []  # list of {label, points_list: [...]}
    for inst in instances:
        centroid = inst["points"].mean(axis=0)
        match = None
        for c in clusters:
            if c["label"] != inst["label"]:
                continue
            c_centroid = np.concatenate(c["points_list"]).mean(axis=0)
            if np.linalg.norm(c_centroid - centroid) < merge_distance:
                match = c
                break
        if match is None:
            clusters.append({"label": inst["label"], "points_list": [inst["points"]]})
        else:
            match["points_list"].append(inst["points"])
    return [{"label": c["label"], "points": np.concatenate(c["points_list"])} for c in clusters]


merged_clusters = merge_instances(raw_instances)
print(f"{len(raw_instances)} instance thô -> {len(merged_clusters)} vật thể sau khi gộp")
for c in merged_clusters:
    print(f"  {c['label']}: {len(c['points'])} điểm, centroid={c['points'].mean(axis=0)}")


## Bước 5: dựng `Detection3D` cho từng cụm (đúng logic mặt tường đã dùng ở `manual_segmentation.py`)

Trục "lên" + mặt tường được ước lượng trên **toàn bộ** point cloud dense của scene (`SCENE_PLY`), không phải trên từng cụm nhỏ - giống hệt cách `manual_segmentation.load_manual_segmentation` làm, chỉ khác nguồn điểm của mỗi opening (backproject từ ảnh, thay vì CloudCompare chọn tay). Phần orient-tường-ra-ngoài + đo width/height dưới đây được viết lại y hệt (không import từ `manual_segmentation.py`, để tránh đụng vào code detection/matching đang chạy đúng) - chỉ 2 hàm dùng chung không đổi là `estimate_up_vector_manhattan`/`extract_wall_planes` từ `plane_fitting.py`.


In [ ]:
from plyfile import PlyData

from sgd_alignment.common.types import Detection3D, PointCloud
from sgd_alignment.detection.plane_fitting import estimate_up_vector_manhattan, extract_wall_planes


def orient_walls_outward(walls, pc, is_outdoor, margin=0.10):
    """Xem docstring gốc trong manual_segmentation.py: quy tắc "ít điểm hơn
    = mặt ngoài" đúng cho scan indoor nhưng NGƯỢC LẠI cho scan outdoor."""
    oriented = {}
    for idx, wall in enumerate(walls):
        wall_point = pc.points[wall.inlier_indices].mean(axis=0)
        normal = wall.normal
        signed = pc.points @ normal - np.dot(normal, wall_point)
        frac_pos = float((signed > margin).mean())
        frac_neg = float((signed < -margin).mean())
        positive_side_is_outward = (frac_pos <= frac_neg) if not is_outdoor else (frac_pos > frac_neg)
        if not positive_side_is_outward:
            normal = -normal
        oriented[idx] = normal
    return oriented


def nearest_wall_normal(centroid, walls, oriented_normals):
    if not walls:
        return None
    best_idx = min(range(len(walls)), key=lambda i: abs(walls[i].signed_distance(centroid[None, :])[0]))
    return oriented_normals[best_idx]


def points_to_detection(points, category, up, wall_normal):
    centroid = points.mean(axis=0)
    if wall_normal is not None:
        normal = wall_normal
    else:
        _, _, vt = np.linalg.svd(points - centroid, full_matrices=False)
        normal = vt[-1]

    v_axis = up - np.dot(up, normal) * normal
    v_axis = v_axis / np.linalg.norm(v_axis)
    u_axis = np.cross(v_axis, normal)
    u_axis = u_axis / np.linalg.norm(u_axis)

    onto_plane = points - np.outer((points - centroid) @ normal, normal)
    centered = onto_plane - centroid
    u = centered @ u_axis
    v = centered @ v_axis
    width = float(u.max() - u.min())
    height = float(v.max() - v.min())
    center = centroid + ((u.max() + u.min()) / 2) * u_axis + ((v.max() + v.min()) / 2) * v_axis

    return Detection3D(category=category, center=center, u_axis=u_axis, v_axis=v_axis,
                        normal=normal, width=width, height=height)


scene_ply = PlyData.read(SCENE_PLY)
scene_vertex = scene_ply["vertex"].data
scene_points = np.stack([scene_vertex["x"], scene_vertex["y"], scene_vertex["z"]], axis=1).astype(np.float64)
scene_pc = PointCloud(points=scene_points)

scene_up = estimate_up_vector_manhattan(scene_pc)
walls = extract_wall_planes(scene_pc, up=scene_up)
oriented_normals = orient_walls_outward(walls, scene_pc, IS_OUTDOOR)
print(f"{len(walls)} mặt tường phát hiện được trên scene")

detections = []
for c in merged_clusters:
    wall_normal = nearest_wall_normal(c["points"].mean(axis=0), walls, oriented_normals)
    detections.append(points_to_detection(c["points"], c["label"], scene_up, wall_normal))

for d in detections:
    print(f"{d.category}: center={d.center}, size={d.width:.2f}x{d.height:.2f}")


## Bước 6: xuất kết quả

- `outputs/<name>_detections.pkl`: `list[Detection3D]`, dùng trực tiếp cho bước matching (`sgd_alignment.matching.sgd.build_sgds` / `align_indoor_outdoor`), thay cho `load_manual_segmentation(...)`.
- `outputs/<name>_openings.ply`: toàn bộ điểm của scene (xám) + các điểm đã backproject theo từng vật thể (tô màu theo category) để kiểm tra bằng mắt trong CloudCompare.


In [ ]:
detections_path = os.path.join(OUTPUT_DIR, f"{OUTPUT_NAME}_detections.pkl")
with open(detections_path, "wb") as f:
    pickle.dump(detections, f)
print("đã lưu:", detections_path)


In [ ]:
from plyfile import PlyElement

def export_colored_ply(path, background_points, clusters, class_colors):
    all_points = [background_points]
    all_colors = [np.full((len(background_points), 3), 160, dtype=np.uint8)]
    for c in clusters:
        color = np.array(class_colors.get(c["label"], (255, 0, 0)), dtype=np.uint8)
        all_points.append(c["points"])
        all_colors.append(np.tile(color, (len(c["points"]), 1)))

    pts = np.concatenate(all_points)
    cols = np.concatenate(all_colors)
    vertex = np.array(
        [(*p, *col) for p, col in zip(pts, cols)],
        dtype=[("x", "f4"), ("y", "f4"), ("z", "f4"), ("red", "u1"), ("green", "u1"), ("blue", "u1")],
    )
    PlyData([PlyElement.describe(vertex, "vertex")], text=False).write(path)


openings_ply_path = os.path.join(OUTPUT_DIR, f"{OUTPUT_NAME}_openings.ply")
export_colored_ply(openings_ply_path, scene_points, merged_clusters, CLASS_COLORS)
print("đã lưu:", openings_ply_path)


## (Tuỳ chọn) Bước 7: chạy matching nếu đã có cả 2 phía indoor/outdoor

Chạy notebook này 2 lần (1 lần `IS_OUTDOOR=False`, 1 lần `IS_OUTDOOR=True`, mỗi lần với `SPARSE_DIR`/`IMAGES_DIR`/`DEPTH_MAPS_DIR`/`SCENE_PLY`/`SELECTED_IMAGES` khớp với bộ ảnh của phía đó) để có đủ 2 file `.pkl`, rồi chạy cell dưới đây để align, giống hệt pipeline `manual_segmentation` đã dùng trước đó.


In [ ]:
from sgd_alignment.matching.alignment import align_indoor_outdoor

with open(os.path.join(OUTPUT_DIR, "indoor_detections.pkl"), "rb") as f:
    indoor_detections = pickle.load(f)
with open(os.path.join(OUTPUT_DIR, "outdoor_detections.pkl"), "rb") as f:
    outdoor_detections = pickle.load(f)

result = align_indoor_outdoor(indoor_detections, outdoor_detections)
print(f"{len(result.matches)} cặp khớp, residuals: {result.residuals}")
for indoor_idx, outdoor_idx, cost in result.matches:
    print(f"  indoor[{indoor_idx}] <-> outdoor[{outdoor_idx}], cost={cost:.3f}")
